In [5]:
from pathlib import Path
out_dir = Path("/Users/matthewwright/Desktop/26-10-19_check_hard_disks")

In [6]:
import re
import os
# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

drive_name = "porsche2_monday_inverse"

base_dir = Path(
    "/Volumes/Rain/original_radar_precip_single"
)

# Text file generated by PowerShell
list_path = Path(
    "/Users/matthewwright/Desktop/26-10-19_check_hard_disks/file_list_precip_single_261019_1513.txt"
)

date_re = re.compile(r"^\d{8}$")
file_re = re.compile(r"^\d{6}\.mdv$", re.IGNORECASE)

# --------------------------------------------------
# READ WINDOWS FILE LIST
# --------------------------------------------------

# PowerShell may have written UTF-8 or UTF-16
with open(list_path, "rb") as f:
    header = f.read(4)

if header.startswith((b"\xff\xfe", b"\xfe\xff")):
    encoding = "utf-16"
elif header.startswith(b"\xef\xbb\xbf"):
    encoding = "utf-8-sig"
else:
    encoding = "utf-8"

with open(list_path, encoding=encoding) as f:
    windows_paths = [line.strip() for line in f if line.strip()]

# Extract day/filename from Windows paths
# Example:
# D:\radar_data\20190206\120000.mdv
# becomes 20190206/120000.mdv

windows_files = set()

for path in windows_paths:

    parts = path.replace("\\", "/").split("/")

    if len(parts) >= 2:
        day, filename = parts[-2:]

        if date_re.fullmatch(day) and file_re.fullmatch(filename):
            windows_files.add(f"{day}/{filename}".lower())

# --------------------------------------------------
# FIND MAC MDV FILES
# --------------------------------------------------

mac_files = set()

for day_dir in base_dir.iterdir():

    if not day_dir.is_dir() or not date_re.fullmatch(day_dir.name):
        continue

    for f in day_dir.iterdir():

        if (
            f.is_file()
            and not f.name.startswith("._")
            and file_re.fullmatch(f.name)
        ):
            mac_files.add(f"{day_dir.name}/{f.name}".lower())

# --------------------------------------------------
# COMPARE
# --------------------------------------------------

missing_from_windows = sorted(mac_files - windows_files)
missing_from_mac = sorted(windows_files - mac_files)

print("FILE COMPARISON")
print("--------------------------------------")
print(f"Windows list MDV files:  {len(windows_files):,}")
print(f"Mac disk MDV files:      {len(mac_files):,}")
print(f"Matching files:          {len(mac_files & windows_files):,}")
print(f"Missing from Windows:    {len(missing_from_windows):,}")
print(f"Missing from Mac:        {len(missing_from_mac):,}")

print("\nFILES ON MAC BUT NOT IN WINDOWS LIST")
print("--------------------------------------")

for f in missing_from_windows[:100]:
    print(f)

if len(missing_from_windows) > 100:
    print(f"... and {len(missing_from_windows)-100:,} more")

drive_out_dir = out_dir / drive_name

drive_out_dir.mkdir(parents=True, exist_ok=True)

(drive_out_dir / "missing_from_windows.txt").write_text(
    "\n".join(missing_from_windows),
    encoding="utf-8"
)

(drive_out_dir / "missing_from_mac.txt").write_text(
    "\n".join(missing_from_mac),
    encoding="utf-8"
)

print(f"\nFull missing-file lists saved to {drive_out_dir}")


FILE COMPARISON
--------------------------------------
Windows list MDV files:  89,701
Mac disk MDV files:      60,492
Matching files:          60,452
Missing from Windows:    40
Missing from Mac:        29,249

FILES ON MAC BUT NOT IN WINDOWS LIST
--------------------------------------
19700101/000120.mdv
19700101/000444.mdv
19700101/000605.mdv
19700101/001005.mdv
19700101/001049.mdv
19700101/001401.mdv
19700101/001449.mdv
19700101/002009.mdv
19700101/002454.mdv
19700101/002939.mdv
19700101/003500.mdv
19700101/003945.mdv
19700101/004506.mdv
19700101/004952.mdv
19700101/005437.mdv
19700101/005958.mdv
19700101/010444.mdv
19700101/011005.mdv
19700101/011450.mdv
19700101/011937.mdv
19700101/012458.mdv
19700101/012944.mdv
19700101/013506.mdv
19700101/013951.mdv
19700101/014437.mdv
19700101/014959.mdv
19700101/015444.mdv
19700101/020005.mdv
19700101/020452.mdv
19700101/020936.mdv
19700101/021457.mdv
19700101/021943.mdv
19700101/022504.mdv
19700101/023444.mdv
19700101/031950.mdv
19700101/033

In [4]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pyart
import re

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

drive_name = "RAIN -- 1hr"
save = False

base_dir = Path(
    # "/Volumes/My Passport/projDir/data/mdv/precip/single"
    "/Volumes/Rain/LiCie/projDir/data/mdv/precip/1hr"
)

VAR = "rate"

date_re = re.compile(r"^\d{8}$")
file_re = re.compile(r"^\d{6}\.mdv$")

# None = all days; or specify {"20190206", "20190227"}
RETRY_DAYS = None

# --------------------------------------------------
# FIND MDV FILES
# --------------------------------------------------

day_dirs = sorted(
    d for d in base_dir.iterdir()
    if d.is_dir()
    and date_re.fullmatch(d.name)
    and (RETRY_DAYS is None or d.name in RETRY_DAYS)
)

files = sorted(
    f
    for d in day_dirs
    for f in d.iterdir()
    if f.is_file()
    and not f.name.startswith("._")
    and file_re.fullmatch(f.name)
)[::10]

print(f"Found {len(day_dirs)} daily directories")
print(f"Found {len(files)} MDV files")

# --------------------------------------------------
# READ MDV RAINFALL
# --------------------------------------------------

rain_values = []

total_pixels = 0
total_valid = 0
total_masked = 0
total_zero = 0
total_positive = 0
total_negative = 0

bad_files = []
zero_byte_files = []
no_positive_files = []

for i, f in enumerate(files):

    # Check empty files
    if f.stat().st_size == 0:
        zero_byte_files.append(str(f))
        continue

    try:
        radar = pyart.io.read_grid_mdv(
            f,
            file_field_names=True
        )

        raw = np.ma.asarray(
            radar.fields[VAR]["data"][0]
        )

        total_pixels += raw.size

        mask = np.ma.getmaskarray(raw)
        total_masked += np.sum(mask)

        values = raw.compressed()
        valid = values[np.isfinite(values)]

        total_valid += valid.size
        total_zero += np.sum(valid == 0)
        total_positive += np.sum(valid > 0)
        total_negative += np.sum(valid < 0)

        positive = valid[valid > 0]

        if positive.size:
            rain_values.append(positive)
        else:
            no_positive_files.append(str(f))

    except Exception as e:
        bad_files.append((str(f), str(e)))
        print(f"BAD FILE: {f.name}: {e}")

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1:,}/{len(files):,} files")

# --------------------------------------------------
# COMBINE POSITIVE RAINFALL VALUES
# --------------------------------------------------

rain_values = (
    np.concatenate(rain_values)
    if rain_values
    else np.array([])
)

# --------------------------------------------------
# STATISTICS
# --------------------------------------------------

print("\nMDV RAINFALL STATISTICS")
print("--------------------------------------")
print(f"Total files:       {len(files):,}")
print(f"Zero-byte files:   {len(zero_byte_files):,}")
print(f"Unreadable files:  {len(bad_files):,}")
print(f"No positive rain:  {len(no_positive_files):,}")
print(f"Total pixels:      {total_pixels:,}")
print(f"Valid pixels:      {total_valid:,}")
print(f"Masked pixels:     {total_masked:,}")
print(f"Zero pixels:       {total_zero:,}")
print(f"Positive pixels:   {total_positive:,}")
print(f"Negative pixels:   {total_negative:,}")

if total_valid:
    print(
        f"Fraction positive among valid: "
        f"{100 * total_positive / total_valid:.3f}%"
    )

if total_pixels:
    print(
        f"Fraction masked: "
        f"{100 * total_masked / total_pixels:.3f}%"
    )

if rain_values.size:
    print("\nPOSITIVE RAINFALL STATISTICS")
    print("--------------------------------------")
    print(f"Minimum positive: {rain_values.min():.6f}")
    print(f"Mean positive:    {rain_values.mean():.6f}")
    print(f"Median positive:  {np.median(rain_values):.6f}")
    print(f"95th percentile:  {np.percentile(rain_values, 95):.6f}")
    print(f"99th percentile:  {np.percentile(rain_values, 99):.6f}")
    print(f"Maximum:          {rain_values.max():.6f}")

# --------------------------------------------------
# SAVE PROBLEM FILE LISTS
# --------------------------------------------------

if save:
    (out_dir / drive_name / "zero_byte_files.txt").write_text(
        "\n".join(zero_byte_files),
        encoding="utf-8"
    )

    (out_dir / drive_name / "unreadable_files.txt").write_text(
        "\n".join(f"{p}\t{e}" for p, e in bad_files),
        encoding="utf-8"
    )

    (out_dir / drive_name / "no_positive_rain_files.txt").write_text(
        "\n".join(no_positive_files),
        encoding="utf-8"
    )

    print(f"\nProblem file lists saved to {out_dir / drive_name}")

# --------------------------------------------------
# HISTOGRAMS
# --------------------------------------------------

if rain_values.size:

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].hist(
        rain_values,
        bins=100,
        color="steelblue",
        edgecolor="none"
    )
    axes[0].set_xlabel("Rainfall rate (mm/h)")
    axes[0].set_ylabel("Pixel count")
    axes[0].set_title("MDV rainfall - linear scale")

    min_rain = rain_values.min()
    max_rain = rain_values.max()

    if min_rain < max_rain:
        log_bins = np.logspace(
            np.log10(min_rain),
            np.log10(max_rain),
            100
        )
    else:
        log_bins = np.array([
            min_rain * 0.9,
            max_rain * 1.1
        ])

    axes[1].hist(
        rain_values,
        bins=log_bins,
        color="steelblue",
        edgecolor="none"
    )

    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Rainfall rate (mm/h)")
    axes[1].set_ylabel("Pixel count")
    axes[1].set_title("MDV rainfall - log scale")

    plt.tight_layout()
    plt.show()

else:
    print("\nNo positive rainfall values found.")



## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly supported
## by the U.S. Department of Energy Office of Science as part of
## the Atmospheric Radiation Measurement (ARM) User Facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119

Found 143 daily directories
Found 2002 MDV files
Processed 100/2,002 files
Processed 200/2,002 files
Processed 300/2,002 files
Processed 400/2,002 files
Processed 500/2,002 files
Processed 600/2,002 files
Processed 700/2,002 files
Processed 800/2,002 files
Processed 900/2,002 files
Processed 1,000/2,002 files
Processed 1,100/2,002 files
Processed 1,200/2,002 files
Processed 1,300/2,002 files
Processed 1,400/2,002 files
Processed 1,500/2,002 files
Processed 1,600/2,002 files
Processed 1,700/2,002 files
Processed 1,800/2,002 files
Processed 1,900/2,002 files
Processed 2,000/2,002 fi